# RL Post-Training using Hugging Face TRL and GRPO

This notebook is based on the TRL GRPO Example, [GRPO Trainer](https://huggingface.co/docs/trl/en/grpo_trainer), to fine-tune a Qwen2.5 0.5B model to answer math questions using the DeepMath-103k dataset and TRL's GRPO (Group Relative Policy Optimization) implementation.

To scale this implementation efficiently across multiple GPUs without changing training logic, we use Ray Train.

This notebook consists of the following steps:
1. [Set up Ray](#set-up)
2. [Quick Summary of GRPO and the DeepMath dataset](#quick-summary)
3. [Running TRL with Ray Train](#training)

Uncomment and run the following line to install all the necessary dependencies. (This notebook is being tested with `transformers==4.57.6`.):


In [ ]:
#! pip install "trl[vllm]" "math_verify" "transformers==4.57.6"

(set-up)=

## Set up Ray

Use `ray.init()` to initialize a local cluster. By default, this cluster contains only the machine you are running this notebook on. You can also run this notebook on an Anyscale cluster.


In [ ]:
import ray

ray.init()

Use `ray.cluster_resources()` to check which resources your cluster has access to.
If you're running this notebook on your local machine or Google Colab, you should see the number of CPU cores and GPUs available to you.

In [3]:
from pprint import pprint

pprint(ray.cluster_resources())

{'CPU': 48.0,
 'GPU': 4.0,
 'accelerator_type:L4': 1.0,
 'anyscale/accelerator_shape:4xL4': 1.0,
 'anyscale/node-group:head': 1.0,
 'anyscale/provider:aws': 1.0,
 'anyscale/region:us-west-2': 1.0,
 'memory': 206158430208.0,
 'node:10.0.201.128': 1.0,
 'node:__internal_head__': 1.0,
 'object_store_memory': 58487200972.0}


This notebook RL post-trains a pretrained Qwen2.5 model to learn how to answer math questions using the DeepMath dataset and [GRPO](https://arxiv.org/abs/2402.03300).
To parallelize this training efficiently across multiple GPUs and/or nodes, we use Ray Train to easily achieve this with fault-tolerance, observability, and more.

You can change these two variables to control whether the training, which we'll explain later, uses CPUs or GPUs, and how many workers to spawn. Each worker will claim one CPU or GPU so make sure to not request more resources than are available. By default, the training runs with one GPU worker.

In [4]:
use_gpu = True  # set this to `False` to run on CPUs
num_workers = 4  # set this to the number of GPUs or CPUs you want to use

(quick-summary)=

## Short Summary of the DeepMath dataset and GRPO algorithm

To test and train our model, we use the DeepMath-103k dataset ([He et al., 2025](https://arxiv.org/abs/2504.11456)), consisting of challenging questions across a wide range of mathematical subjects including Algebra, Calculus, Number Theory, Geometry, Probability, and Discrete Mathematics. These questions are more heavily distributed towards Levels 5 to 9 compared to alternative datasets making them highly complex and difficult to solve without specialist training.

The questions consist of a question prompt for the model and a solution in a LaTeX format to provide a consistent format that complex mathematical equations can be expressed.

In [6]:
from datasets import load_dataset

dataset = load_dataset("trl-lib/DeepMath-103K", split="train").shuffle(seed=123).select(range(10))

pprint(dataset[0])
pprint(dataset[1])

{'prompt': [{'content': 'Determine the number of 19th power residues modulo '
                        '229.',
             'role': 'user'}],
 'solution': '$12$'}
{'prompt': [{'content': 'Determine the norm \\( \\|T\\| \\) of the linear '
                        'operator \\( T: \\ell^2 \\rightarrow \\ell^2 \\) '
                        'given by\n'
                        '\\[\\begin{cases} (Tx)_1 = 0, \\\\ (Tx)_n = -x_n + '
                        '\\alpha x_{n+1}, \\quad n \\geq 2, \\end{cases}\\]\n'
                        'where \\( \\alpha \\in \\mathbb{C} \\). Provide the '
                        'tightest possible bounds for \\( \\|T\\| \\).',
             'role': 'user'}],
 'solution': '$1 + |\\alpha|$'}


To RL fine-tune our model to answer math questions, we use TRL's GRPO (Group Relative Policy Optimization) implementation that was proposed by [Shao et al., 2024](https://arxiv.org/abs/2402.03300).

For a deeper summary of the algorithm, the [TRL GRPO page](https://huggingface.co/docs/trl/en/grpo_trainer#looking-deeper-into-the-grpo-method) provides a more detailed description.

(training)=

## Using Hugging Face TRL (Transformer Reinforcement Learning) with Ray Train

In comparison to supervised pre-training where models are optimised to minimise the error for their next token, RL-based post-training aims to maximise their reward from a prompt. Therefore, it's crucial to define a reward function to measure the success and to train a model.

In this notebook, we use the `trl.rewards.accuracy_reward` function to check whether the model's answer matches the dataset's answer. Because the answers are written in LaTeX, both responses must be parsed before they can be compared. The default `trl.rewards.accuracy_reward` implementation applies timeouts to the `parse` and `verify` functions, which are incompatible with Ray. The version defined below disables these timeouts by setting `parsing_timeout=0` and `timeout_seconds=0`.

In [7]:
from latex2sympy2_extended import NormalizationConfig
from math_verify import LatexExtractionConfig, parse, verify

def accuracy_reward(completions, solution, **kwargs):
    """Reward function that checks mathematical accuracy.

    This is a copy of `trl.rewards.accuracy_reward`.
    The only difference is `parse(..., parsing_timeout=0)` and `verify(..., timeout_seconds=0)`
    to avoid `signal.alarm()` issues with ray.
    """
    contents = [completion[0]["content"] for completion in completions]
    rewards = []
    for content, sol in zip(contents, solution, strict=True):
        gold_parsed = parse(sol, parsing_timeout=0)
        if len(gold_parsed) != 0:
            # We require the answer to be provided in correct latex (no malformed operators)
            answer_parsed = parse(
                content,
                extraction_config=[
                    LatexExtractionConfig(
                        normalization_config=NormalizationConfig(units=True),
                        # Ensures that boxed is tried first
                        boxed_match_priority=0,
                        try_extract_without_anchor=False,
                    )
                ],
                extraction_mode="first_match",
                parsing_timeout=0,
            )
            reward = float(verify(gold_parsed, answer_parsed, timeout_seconds=0))
        else:
            # If the gold solution cannot be parsed, we assign `None` to skip this example
            reward = None
        rewards.append(reward)

    return rewards

Now, we need to build our training function that is parallelized to all the workers. We build a `GRPOTrainer` which implements the GRPO algorithm.

We use `vllm_mode="colocate"`, where each worker runs a vLLM instance that consumes about 30% of the GPU memory. The alternative `"server"` mode dedicates one worker to generation with vLLM while the others perform learning, but this introduces inter-GPU communication overhead that can reduce throughput. See [this blog post](https://huggingface.co/blog/vllm-colocate) for more details.

Two Ray Train-specific additions integrate the HF Trainer into the Ray Train ecosystem:

- **`RayTrainReportCallback`** — hooks into the HF Trainer's logging to forward metrics and checkpoints to Ray Train after each training step. This is what populates the `Result` object returned by `trainer.fit()` with metrics like `reward` and the best checkpoint.
- **`prepare_trainer`** — configures the HF Trainer for distributed execution across Ray workers. It disables HF's built-in distributed setup so that Ray Train controls the process group instead, and ensures correct device placement on each worker.

In [8]:
from ray.train.huggingface.transformers import RayTrainReportCallback, prepare_trainer
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset

def train_func(config):
    # Datasets
    dataset = load_dataset("trl-lib/DeepMath-103K", split="train").shuffle(seed=123).select(range(100))


    # Use vllm_mode="colocate" to allow easy scaling as each GPU handles their own vLLM instance
    training_args = GRPOConfig(
        per_device_train_batch_size=4, 
        use_vllm=True, vllm_mode="colocate", 
        vllm_gpu_memory_utilization=0.3,
        num_train_epochs=2,
    )

    # GRPO Trainer
    trainer = GRPOTrainer(
        model="Qwen/Qwen2.5-0.5B",
        args=training_args,
        reward_funcs=accuracy_reward,
        train_dataset=dataset,
    )

    # Report metrics and checkpoints to Ray Train
    trainer.add_callback(RayTrainReportCallback())

    # Prepare your trainer for Ray Data integration
    trainer = prepare_trainer(trainer)

    # Start Training
    trainer.train()

With your `train_func` complete, you can now instantiate the {class}`~ray.train.torch.TorchTrainer`. Aside from calling the function, set the `scaling_config`, which controls the number of workers and resources used, and the `run_config` to configure checkpointing.

The `CheckpointConfig` controls how Ray Train saves and selects checkpoints during training:

- **`num_to_keep=1`** — only the single best checkpoint is retained on disk, saving storage.
- **`checkpoint_score_attribute="reward"`** — checkpoints are ranked by the `"reward"` metric, which is the mean reward across the batch as reported by `RayTrainReportCallback`.
- **`checkpoint_score_order="max"`** — higher reward is better, so Ray Train keeps the checkpoint with the highest reward seen across all training steps.

In [9]:
from ray.train.torch import TorchTrainer
from ray.train import RunConfig, ScalingConfig, CheckpointConfig

trainer = TorchTrainer(
    train_func,
    scaling_config=ScalingConfig(num_workers=num_workers, use_gpu=use_gpu),
    run_config=RunConfig(
        checkpoint_config=CheckpointConfig(
            num_to_keep=1,
            checkpoint_score_attribute="reward",
            checkpoint_score_order="max",
        ),
    ),
)

Finally, call the `fit` method to start training with Ray Train. Save the `Result` object to a variable so you can access metrics and checkpoints.

In [10]:
result = trainer.fit()

(TrainController pid=18655) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=18655) Attempting to start training worker group of size 4 with the following resources: [{'GPU': 1}] * 4
(RayTrainWorker pid=18765) Setting up process group for: env:// [rank=0, world_size=4]
(TrainController pid=18655) Started training worker group of size 4: 
(TrainController pid=18655) - (ip=10.0.201.128, pid=18765) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=18655) - (ip=10.0.201.128, pid=18766) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=18655) - (ip=10.0.201.128, pid=18767) world_rank=2, local_rank=2, node_rank=0
(TrainController pid=18655) - (ip=10.0.201.128, pid=18768) world_rank=3, local_rank=3, node_rank=0
(TrainController pid=18655) [State Transition] SCHEDULING -> RUNNING.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  7.04it/s]
Loading safet

(RayTrainWorker pid=18765) {'loss': 0.0, 'grad_norm': 0.0, 'learning_rate': 9.1e-07, 'num_tokens': 47983.0, 'completions/mean_length': 219.79375, 'completions/min_length': 68.4, 'completions/max_length': 256.0, 'completions/clipped_ratio': 0.73125, 'completions/mean_terminated_length': 109.41024017333984, 'completions/min_terminated_length': 42.8, 'completions/max_terminated_length': 181.8, 'rewards/accuracy_reward/mean': 0.0, 'rewards/accuracy_reward/std': 0.0, 'reward': 0.0, 'reward_std': 0.0, 'frac_reward_zero_std': 1.0, 'sampling/sampling_logp_difference/mean': 0.019891180098056793, 'sampling/sampling_logp_difference/max': 0.4741410255432129, 'sampling/importance_sampling_ratio/min': 0.22699655443429947, 'sampling/importance_sampling_ratio/mean': 0.911529678106308, 'sampling/importance_sampling_ratio/max': 2.090355694293976, 'entropy': 1.8360318005084992, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/regi

 11%|████▌                                    | 11/100 [00:35<03:54,  2.64s/it]
(RayTrainWorker pid=18765) [RANK 0] All reward functions returned None for the following kwargs:
(RayTrainWorker pid=18765) {'solution': 'Yes', 'prompt': [{'content': 'Given two positive real numbers $A$ and $l$, such that $4\\pi A \\leq l^2$, determine whether it is always possible to find a plane, simple, closed curve with length $l$ that encloses a region with area $A$. Provide a justification for your answer.', 'role': 'user'}], 'completion': [{'role': 'assistant', 'content': 'To determine whether it is always possible to find a simple closed curve with length $l$ that encloses a region with area $A$ given that $4\\pi A \\leq l^2$, we need to use some fundamental concepts from geometry and calculus.\n\n### Step-by-Step Reasoning:\n\n1. **Understanding the constraint:**\n   The constraint given is $4\\pi A \\leq l^2$. This means the area of the region enclosed by the curve and the actual circle formed by

(RayTrainWorker pid=18765) {'loss': -0.0005, 'grad_norm': 2.4105613231658936, 'learning_rate': 8.1e-07, 'num_tokens': 97251.0, 'completions/mean_length': 223.275, 'completions/min_length': 61.7, 'completions/max_length': 256.0, 'completions/clipped_ratio': 0.73125, 'completions/mean_terminated_length': 143.5395248413086, 'completions/min_terminated_length': 61.7, 'completions/max_terminated_length': 221.3, 'rewards/accuracy_reward/mean': 0.00625, 'rewards/accuracy_reward/std': 0.025, 'reward': 0.00625, 'reward_std': 0.025, 'frac_reward_zero_std': 0.95, 'sampling/sampling_logp_difference/mean': 0.020900878682732583, 'sampling/sampling_logp_difference/max': 0.3208913803100586, 'sampling/importance_sampling_ratio/min': 0.2180515393614769, 'sampling/importance_sampling_ratio/mean': 0.9663178563117981, 'sampling/importance_sampling_ratio/max': 2.1550982117652895, 'entropy': 1.8047699689865113, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/hi

 30%|████████████▎                            | 30/100 [01:22<02:52,  2.46s/it]


...

 91%|█████████████████████████████████████▎   | 91/100 [03:53<00:22,  2.53s/it]
(RayTrainWorker pid=18765) [RANK 0] All reward functions returned None for the following kwargs:
(RayTrainWorker pid=18765) {'solution': 'Yes', 'prompt': [{'content': 'Determine whether there exists an $L_1$ function on the interval $[0,1]$ whose essential supremum is attained on an uncountable set.', 'role': 'user'}], 'completion': [{'role': 'assistant', 'content': 'The statement "$L_1$ function is an essential supremum of a countable set" is equivalent to $" \\forall A, \\exists M\\in \\mathbb{R}^": \\underline{\\int}^1_A \\vert f(x)\\ {dx}\\le\\ M$" If we divide by $1$, the condition can be simplified to " $" \\forall A, \\exists M\\in \\mathbb{R}^": \\small \\int_A \\vert f(x)\\  dt\\le\\ M" $" "$Of" "My line"\n(UIAlertActionctionsystem\nYou are a helpful assistant.翙\n翙user\nGiven $a>0$. Define $R(x) := \\frac{1}{b+c\\ x^2}$. Show that "$R(x)$ obtains its maximum at $x = 0$" IIDAEICKtmsysa May 19, 2019 

(RayTrainWorker pid=18765) {'loss': 0.0, 'grad_norm': 0.0, 'learning_rate': 1e-08, 'num_tokens': 504887.0, 'completions/mean_length': 229.25625, 'completions/min_length': 75.2, 'completions/max_length': 256.0, 'completions/clipped_ratio': 0.8, 'completions/mean_terminated_length': 107.65833435058593, 'completions/min_terminated_length': 49.6, 'completions/max_terminated_length': 174.7, 'rewards/accuracy_reward/mean': 0.0, 'rewards/accuracy_reward/std': 0.0, 'reward': 0.0, 'reward_std': 0.0, 'frac_reward_zero_std': 1.0, 'sampling/sampling_logp_difference/mean': 0.01890042582526803, 'sampling/sampling_logp_difference/max': 0.34836219549179076, 'sampling/importance_sampling_ratio/min': 0.19625091254711152, 'sampling/importance_sampling_ratio/mean': 0.9524671196937561, 'sampling/importance_sampling_ratio/max': 1.912125289440155, 'entropy': 1.4707762956619264, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_m

(RayTrainWorker pid=18766) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/ray/ray_results/ray_train_run-2026-03-12_09-07-13/checkpoint_2026-03-12_09-13-20.499676)
(RayTrainWorker pid=18766) Reporting training result 1: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=/home/ray/ray_results/ray_train_run-2026-03-12_09-07-13/checkpoint_2026-03-12_09-13-20.499676), metrics={'loss': 0.0, 'grad_norm': 0.0, 'learning_rate': 1e-08, 'num_tokens': 504887.0, 'completions/mean_length': 229.25625, 'completions/min_length': 75.2, 'completions/max_length': 256.0, 'completions/clipped_ratio': 0.8, 'completions/mean_terminated_length': 107.65833435058593, 'completions/min_terminated_length': 49.6, 'completions/max_terminated_length': 174.7, 'rewards/accuracy_reward/mean': 0.0, 'rewards/accuracy_reward/std': 0.0, 'reward': 0.0, 'reward_std': 0.0, 'frac_reward_zero_std': 1.0, 'sampling/sampling_logp_difference/mean': 0.01890042582526803, 'sampling/sampling_logp_dif

(RayTrainWorker pid=18765) {'train_runtime': 277.388, 'train_samples_per_second': 0.721, 'train_steps_per_second': 0.361, 'train_loss': -0.00016631588339805604, 'epoch': 2.0}


100%|████████████████████████████████████████| 100/100 [04:37<00:00,  2.78s/it]
(TrainController pid=18655) [State Transition] RUNNING -> SHUTTING_DOWN.
(TrainController pid=18655) [State Transition] SHUTTING_DOWN -> FINISHED.


You can use the returned `Result` object to access metrics and the Ray Train `Checkpoint` associated with the last iteration.

In [11]:
result

Result(metrics={'loss': 0.0, 'grad_norm': 0.0, 'learning_rate': 1e-08, 'num_tokens': 504887.0, 'completions/mean_length': 229.25625, 'completions/min_length': 75.2, 'completions/max_length': 256.0, 'completions/clipped_ratio': 0.8, 'completions/mean_terminated_length': 107.65833435058593, 'completions/min_terminated_length': 49.6, 'completions/max_terminated_length': 174.7, 'rewards/accuracy_reward/mean': 0.0, 'rewards/accuracy_reward/std': 0.0, 'reward': 0.0, 'reward_std': 0.0, 'frac_reward_zero_std': 1.0, 'sampling/sampling_logp_difference/mean': 0.01890042582526803, 'sampling/sampling_logp_difference/max': 0.34836219549179076, 'sampling/importance_sampling_ratio/min': 0.19625091254711152, 'sampling/importance_sampling_ratio/mean': 0.9524671196937561, 'sampling/importance_sampling_ratio/max': 1.912125289440155, 'entropy': 1.4707762956619264, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, '

## See also

* {doc}`Ray Train Examples <../../examples>` for more use cases
* {ref}`Ray Train User Guides <train-user-guides>` for how-to guides

## Summary

This notebook demonstrated how to use Ray Train to scale RL post-training of a Qwen2.5 0.5B model across multiple GPUs using TRL's GRPO implementation and the DeepMath-103k dataset.

The key Ray Train integration points were:

- **`RayTrainReportCallback`** — forwards HF Trainer metrics and checkpoints to Ray Train after each step.
- **`prepare_trainer`** — configures the HF Trainer to run distributed training under Ray's process group.
- **`TorchTrainer`** — launches `train_func` on all workers with the configured resources.
- **`CheckpointConfig`** — automatically tracks and retains the best checkpoint based on the `"reward"` metric.

From here, you can:
- Scale to more GPUs or multiple nodes by increasing `num_workers` in `ScalingConfig`.
- Swap in a different base model or dataset by updating the `model` argument and `load_dataset` call.
- Experiment with different reward functions to train for other tasks.